## DATASET EXPLORATION


Goal is to understand data

IMPORTS

In [1]:
import sys
import os

# 1. Récupère le chemin absolu du dossier courant du notebook
current_dir = os.getcwd()

# 2. Récupère le chemin du dossier parent (le dossier 'Segmentation')
parent_dir = os.path.dirname(current_dir)

# 3. Ajoute ce dossier parent au path de Python
sys.path.append(parent_dir)

print(f"Dossier ajouté au path : {parent_dir}")
print("Tu peux maintenant importer config, data, et models.")


import config
import data.dataset as dataset
from models.lightning import HemorrhageModel






Dossier ajouté au path : /store/home/tibia/Projet_Hemorragie/Seg_hemorragie/Segmentation
Tu peux maintenant importer config, data, et models.


/home/tibia/miniconda3/envs/hemorragie-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
cfg=config.CONFIG

train_files = dataset.get_data_files(f"{cfg['dataset']['dataset_dir']}/train/img",
                                            f"{cfg['dataset']['dataset_dir']}/train/seg")
val_files = dataset.get_data_files(f"{cfg['dataset']['dataset_dir']}/val/img",
                                        f"{cfg['dataset']['dataset_dir']}/val/seg")

        

# Label vide ?
cfg=config.CONFIG

train_files = dataset.get_data_files(f"{cfg['dataset']['dataset_dir']}/train/img",
                                            f"{cfg['dataset']['dataset_dir']}/train/seg")
val_files = dataset.get_data_files(f"{cfg['dataset']['dataset_dir']}/val/img",
                                        f"{cfg['dataset']['dataset_dir']}/val/seg")
test_files = dataset.get_data_files(f"{cfg['dataset']['dataset_dir']}/test/img",
                                        f"{cfg['dataset']['dataset_dir']}/test/seg")

import nibabel as nib
import numpy as np
from tqdm import tqdm # Pour avoir une barre de progression sympa

def get_empty_labels(data_files):
    """
    Parcourt une liste de dictionnaires et retourne les chemins des masques vides.
    On suppose que data_files est une liste du type: [{'image': '...', 'label': '...'}, ...]
    """
    empty_masks = []
    
    # On itère avec tqdm pour voir l'avancement
    for item in tqdm(data_files, desc="Analyse des masques"):
        label_path = item['seg']
        
        # Charger uniquement les données matricielles du NIfTI
        # get_fdata() convertit en float et charge le volume en mémoire
        mask_data = nib.load(label_path).get_fdata()
        
        # Un masque est vide si sa valeur maximale est 0
        if np.max(mask_data) == 0:
            empty_masks.append(label_path)
            
    return empty_masks

# --- Exécution ---
print("Vérification du set d'entraînement...")
empty_in_train = get_empty_labels(train_files)
print(f"-> {len(empty_in_train)} masques vides trouvés sur un total de {len(train_files)} patients.\n")

print("Vérification du set de validation...")
empty_in_val = get_empty_labels(val_files)
print(f"-> {len(empty_in_val)} masques vides trouvés sur un total de {len(val_files)} patients.")

empty_in_test = get_empty_labels(test_files)
print(f"-> {len(empty_in_test)} masques vides trouvés sur un total de {len(test_files)} patients.")
# Optionnel : Afficher quelques exemples de fichiers vides
if empty_in_train:
    print("\nExemples de masques vides (Train) :")
    for path in empty_in_train[:3]:
        print(f" - {path}")


Vérification du set d'entraînement...


Analyse des masques: 100%|██████████| 165/165 [00:09<00:00, 17.16it/s]


-> 9 masques vides trouvés sur un total de 165 patients.

Vérification du set de validation...


Analyse des masques: 100%|██████████| 35/35 [00:02<00:00, 14.66it/s]


-> 0 masques vides trouvés sur un total de 35 patients.


Analyse des masques: 100%|██████████| 38/38 [00:02<00:00, 15.44it/s]

-> 0 masques vides trouvés sur un total de 38 patients.

Exemples de masques vides (Train) :
 - /home/tibia/Projet_Hemorragie/Datasets/mbh/Split_Final_Stratified/train/seg/MBH_SEG_2025_LLG_2025_06_12_ID_6caf8d60_ID_3634a47122.nii.gz
 - /home/tibia/Projet_Hemorragie/Datasets/mbh/Split_Final_Stratified/train/seg/MBH_SEG_2025_LLG_2025_06_12_ID_8d914da6_ID_b4eb9c8234.nii.gz
 - /home/tibia/Projet_Hemorragie/Datasets/mbh/Split_Final_Stratified/train/seg/MBH_SEG_2025_LLG_2025_06_12_ID_d039497d_ID_da6cc2290f.nii.gz


In [3]:
def count_voxels_per_label(seg_filepath):
    """
    Charge un fichier NIfTI de segmentation, le convertit en tableau NumPy,
    et compte le nombre de voxels pour chaque valeur de label.
    """
    print(f"\nChargement du masque: {seg_filepath}")
    
    # 1. Charger le fichier NIfTI
    try:
        nifti_img = nib.load(seg_filepath)
        
        # 2. Obtenir les données sous forme de tableau NumPy
        # Note : On force souvent le chargement de l'intégralité du tableau en mémoire.
        seg_data = nifti_img.get_fdata() 
        
    except Exception as e:
        return f"Erreur lors du chargement de {seg_filepath} : {e}"

    # 3. Mettre le tableau à plat et compter les occurrences
    # Les masques de segmentation sont souvent en entiers
    seg_data_flat = seg_data.astype(np.int32).flatten()
    
    # La librairie collections.Counter est la façon la plus simple de compter
    # les occurrences d'éléments dans une liste/tableau.
    voxel_counts = Counter(seg_data_flat)
    
    return voxel_counts

In [7]:
import nibabel as nib
import numpy as np
from collections import Counter

test_files = get_data_files(f"{DATASET_DIR}/train/img", f"{DATASET_DIR}/train/seg") 
selected_file_path = test_files[2]["seg"] 
print("Comptage des voxels:")
counts = count_voxels_per_label(selected_file_path)

# --- Affichage des résultats ---
for label, count in counts.items():
     print(f"  Label {label}: {count} voxels")



Comptage des voxels:

Chargement du masque: /home/tibia/Projet_Hemorragie/mbh_seg/nii/train/seg/ID_02f779fb_ID_c4d7f33559.nii.gz
  Label 0: 8091395 voxels
  Label 2: 32648 voxels
  Label 3: 2421 voxels


Création données de tests:

In [3]:
# Transforms
test_transforms = T.Compose([
    T.LoadImaged(keys=["image", "seg"]),
])

# Dataset / DataLoader
test_dataset = PersistentDataset(
    test_files,
    transform=test_transforms,
    cache_dir=os.path.join(SAVE_DIR, "cache_train")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=1  
)


In [23]:
print("\n" + "="*60)
print("EXPLORATION DES MÉTADONNÉES")
print("="*60)


for i, batch in enumerate(test_loader):
    print(f"\n--- BATCH {i} ---")
    print(f"keys in batch {list(batch.keys())}")
 
    for key, value in batch.items():
        print(f"\n[{key}]:")
        print(f"  Type: {type(value)}")

        if hasattr(value, 'shape'):
            print(f"  Shape: {value.shape}")
        
        if hasattr(value, 'dtype'):
            print(f"  Dtype: {value.dtype}")

        if isinstance(value, dict):
            print(f"Dict ")

        elif hasattr(value, 'meta') and hasattr(value.meta, 'keys'):
            print(f"  Métadonnées disponibles: {list(value.meta.keys())}")

        elif hasattr(value, '__dict__'):
            
            attrs = [attr for attr in dir(value) if not attr.startswith('_')]
            if attrs:
                print(f"  Attributs disponibles: {attrs}")
    
            
        







    if i == 0:
        break


EXPLORATION DES MÉTADONNÉES


/home/tibia/Projet_Hemorragie/hemorragie-env/lib/python3.12/site-packages/monai/data/dataset.py:374: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(hashfile


--- BATCH 0 ---
keys in batch ['image', 'seg']

[image]:
  Type: <class 'monai.data.meta_tensor.MetaTensor'>
  Shape: torch.Size([1, 512, 512, 32])
  Dtype: torch.float32
  Métadonnées disponibles: ['scl_slope', 'qform_code', 'quatern_b', 'qoffset_x', 'filename_or_obj', 'extents', 'cal_max', 'dim', original_affine, 'qoffset_z', 'sizeof_hdr', 'srow_z', 'srow_y', spatial_shape, 'slice_end', 'glmin', 'as_closest_canonical', 'quatern_d', 'intent_code', 'pixdim', 'toffset', 'dim_info', 'intent_p2', 'session_error', 'slice_start', affine, 'datatype', 'glmax', 'intent_p3', space, 'qoffset_y', 'slice_duration', 'bitpix', 'xyzt_units', 'slice_code', 'sform_code', 'srow_x', 'intent_p1', original_channel_dim, 'vox_offset', 'scl_inter', 'quatern_c', 'cal_min']

[seg]:
  Type: <class 'monai.data.meta_tensor.MetaTensor'>
  Shape: torch.Size([1, 512, 512, 32])
  Dtype: torch.float32
  Métadonnées disponibles: ['scl_slope', 'qform_code', 'quatern_b', 'qoffset_x', 'filename_or_obj', 'extents', 'cal_ma

In [24]:
for i, batch in enumerate(test_loader):
    print(f"Filename image: {batch['image'].meta['filename_or_obj']}")
    print(f"Filename seg: {batch['seg'].meta['filename_or_obj']}")
    print(f"Type filename: {type(batch['image'].meta['filename_or_obj'])}")
    if i == 0:
        break

/home/tibia/Projet_Hemorragie/hemorragie-env/lib/python3.12/site-packages/monai/data/dataset.py:374: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(hashfile

Filename image: ['/home/tibia/Projet_Hemorragie/mbh_seg/nii/test/img/ID_0219ef88_ID_e5c1a31210.nii.gz']
Filename seg: ['/home/tibia/Projet_Hemorragie/mbh_seg/nii/test/seg/ID_0219ef88_ID_e5c1a31210.nii.gz']
Type filename: <class 'list'>


Transfert learning

In [3]:
DATASET_DIR = '/home/tibia/Projet_Hemorragie/Seg_hemorragie/split_MONAI'
SAVE_DIR = "/home/tibia/Projet_Hemorragie/trleaning"
CHECKPOINT_PATH = "/home/tibia/Projet_Hemorragie/v1_log_test/lightning_logs/version_0/checkpoints/epoch=999-step=156000.ckpt"
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
import torch
from monai.networks.nets import UNet
import pytorch_lightning as pl

checkpoint = torch.load(CHECKPOINT_PATH,weights_only=True)
print(f"Checkpoint keys: {list(checkpoint.keys())}")
print(f"number of epochs: {checkpoint['epoch']}")  
print(f"number of steps: {checkpoint['global_step']}")  
print(f"model state dict keys: {(checkpoint['state_dict'].keys())}")
# afficher un poids
old_state_dict = checkpoint["state_dict"]


class HemorrhageModel(pl.LightningModule):
    def __init__(self, num_steps):
        super().__init__()
        self.num_steps = num_steps
        self.model = UNet(
            spatial_dims=3,
            in_channels=1,
            out_channels=6,
            channels=(32, 64, 128, 256, 320, 320),
            strides=(2, 2, 2, 2, 2),
            num_res_units=2,
        )

new_model= HemorrhageModel(num_steps=156000)
new_state_dict = new_model.state_dict()  # Obtenir le state_dict du nouveau modèle
for name, param in old_state_dict.items():
    if name in new_state_dict:
        if param.shape == new_state_dict[name].shape:
            new_state_dict[name].copy_(param)
            print(f"Poids transférés pour la couche : {name}")
        else:
            print(f"Poids non transférés pour {name} (dimensions incompatibles : {param.shape} vs {new_state_dict[name].shape})")
    else:
        print(f"Poids {name} non trouvé dans le nouveau modèle")
      

print(list(old_state_dict.keys())[-10:])

Checkpoint keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers']
number of epochs: 999
number of steps: 156000
model state dict keys: odict_keys(['model.model.0.conv.unit0.conv.weight', 'model.model.0.conv.unit0.conv.bias', 'model.model.0.conv.unit0.adn.A.weight', 'model.model.0.conv.unit1.conv.weight', 'model.model.0.conv.unit1.conv.bias', 'model.model.0.conv.unit1.adn.A.weight', 'model.model.0.residual.weight', 'model.model.0.residual.bias', 'model.model.1.submodule.0.conv.unit0.conv.weight', 'model.model.1.submodule.0.conv.unit0.conv.bias', 'model.model.1.submodule.0.conv.unit0.adn.A.weight', 'model.model.1.submodule.0.conv.unit1.conv.weight', 'model.model.1.submodule.0.conv.unit1.conv.bias', 'model.model.1.submodule.0.conv.unit1.adn.A.weight', 'model.model.1.submodule.0.residual.weight', 'model.model.1.submodule.0.residual.bias', 'model.model.1.submodule.1.submodule.0.conv.unit0.conv.weight', 'model.mode